# DISSERTAÇÃO DE MESTRADO

@author: Guilherme Nogueira

### IMPORTAÇÕES

In [1]:
# Gerais
import numpy as np
import pandas as pd

# Gráficos
import numpy as np
import plotly.io as pio
import matplotlib.pyplot as plt

# Localização
import locale
locale.setlocale(locale.LC_ALL, 'pt_BR.UTF-8')

'pt_BR.UTF-8'

## LENDO A MIP E EXTRAINDO VARIÁVEIS

### Leitura das Abas

In [2]:
ARQUIVO = "..\\data\\mips\\48008000133202507_Ian_Anexo.xlsx"

raw_monetario    = pd.read_excel(ARQUIVO, sheet_name="Fluxos Monetários",  header=None)
raw_energia      = pd.read_excel(ARQUIVO, sheet_name="Fluxos Energia",     header=None)
raw_en_demfinal  = pd.read_excel(ARQUIVO, sheet_name="Energia Demanda Final", header=None)
raw_emp_setorial = pd.read_excel(ARQUIVO, sheet_name="Emprego Setorial",   header=None)
raw_emprego      = pd.read_excel(ARQUIVO, sheet_name="Emprego",            header=None)
raw_energia_tab  = pd.read_excel(ARQUIVO, sheet_name="Energia",            header=None)

### Nome dos Setores

In [3]:
N = 73  # número de setores

# Linha 5 (índice 5): nomes; linha 4 (índice 4): códigos S01..S73
# Colunas 3..75 correspondem aos setores
sector_names = list(raw_monetario.iloc[5, 3:3+N])
sector_codes = list(raw_monetario.iloc[4, 3:3+N])
sectors = pd.Index(sector_codes, name="setor")

### Matriz Z  

- Fluxos Intermediários 
- Tamanho: N x N

In [ ]:
Z_raw = raw_monetario.iloc[7:7+N, 3:3+N].values.astype(float)
Z = pd.DataFrame(Z_raw, index=sectors, columns=sectors)

### Demanda Final 

- (colunas 77..83)
- 77 = Exportações
- 78 = Governo
- 79 = ISFLSF
- 80 = Famílias
- 81 = FBCF
- 82 = Variação estoque
- 83 = Demanda Final Total

In [6]:
df_names = ["Exportacoes", "Consumo_Governo", "Consumo_ISFLSF",
            "Consumo_Familias", "FBCF", "Variacao_Estoque"]

# Apenas os componentes individuais vêm da MIP (não deriváveis)
y_raw = raw_monetario.iloc[7:7+N, 77:77+6].values.astype(float)
y = pd.DataFrame(y_raw, index=sectors, columns=df_names)

# valor da produção total (col 84)
x_raw = raw_monetario.iloc[7:7+N, 84].values.astype(float)
x = pd.Series(x_raw, index=sectors, name="Producao_Total")

### Valor Adicionado  

- linhas abaixo dos setores
- Referência pelas labels da col 0

In [7]:
# Mapeamento: código -> linha no DataFrame
va_map = {
    "R10": "Remuneracoes",
    "R11": "Salarios",
    "R12": "Contrib_Sociais_Efetivas",
    "R13": "Contrib_Sociais_Imputadas",
    "N2":  "EOB_Rendimento_Misto",
    "N0":  "VA_Custo_Fatores",
    "R22": "Outros_Impostos_Producao",
    "R32": "Outros_Subsidios_Producao",
    "P10": "Valor_Producao",
}

va = {}
for i in range(len(raw_monetario)):
    cod = raw_monetario.iloc[i, 0]
    if pd.notna(cod) and cod in va_map:
        vals = raw_monetario.iloc[i, 3:3+N].values.astype(float)
        va[va_map[cod]] = pd.Series(vals, index=sectors)

VA = pd.DataFrame(va)

### Demanda Final Total

In [ ]:
# f: demanda final = soma dos componentes individuais (identidade contábil)
y["Demanda_Final_Total"] = y.sum(axis=1)

f = y["Demanda_Final_Total"].rename("Demanda_Final")

### Consumo Intermediário

In [9]:
# Nota: CI da MIP inclui impostos sobre produtos (ICMS, IPI, outros IIL)
# que estão fora de Z — por isso CI é lido diretamente da MIP abaixo.

# CI: lido da MIP pois inclui impostos não contidos em Z
for i in range(len(raw_monetario)):
    lab = str(raw_monetario.iloc[i, 1]).strip()
    if "CONSUMO INTERMEDIÁRIO" in lab and "Total" not in lab:
        CI = pd.Series(raw_monetario.iloc[i, 3:3+N].values.astype(float),
                       index=sectors, name="CI_Total")
        break

# VAB/PIB = x - CI  (valor adicionado bruto = produção - consumo intermediário)
VA["VAB_PIB"] = (x - CI).rename("VAB_PIB")

### Fator trabalho

In [10]:
# Fator trabalho (última linha da aba)
for i in range(len(raw_monetario)):
    lab = str(raw_monetario.iloc[i, 1]).strip()
    if "Fator trabalho" in lab:
        emp_vals = raw_monetario.iloc[i, 3:3+N].values.astype(float)
        emprego_mip = pd.Series(emp_vals, index=sectors, name="Ocupacoes")
        break

### Matriz A  e  Leontief 

- (I - A)^-1

In [11]:
x_inv = np.where(x.values != 0, 1.0 / x.values, 0.0)
A_raw = Z.values * x_inv[np.newaxis, :] # A = Z . diag(x)^-1
A = pd.DataFrame(A_raw, index=sectors, columns=sectors)

I = np.eye(N)
L_raw = np.linalg.inv(I - A_raw)
L = pd.DataFrame(L_raw, index=sectors, columns=sectors)

### Matriz de Energia  e  (produtos x setores)

In [12]:
# Linha 3: cabeçalho produtos; linha 5: unidade; linhas 6..11: dados
energy_products = list(raw_energia.iloc[3:5, 1].ffill())
en_labels = raw_energia.iloc[6:, 1].values          # nomes dos produtos
en_data   = raw_energia.iloc[6:, 3:3+N].values.astype(float)
en_rows   = [str(v).strip() for v in en_labels if pd.notna(v)]

E_raw = raw_energia.iloc[6:, 3:3+N].values.astype(float)
en_index = pd.Index([str(v).strip() for v in raw_energia.iloc[6:, 1]], name="produto_energia")
E = pd.DataFrame(E_raw, index=en_index, columns=sectors)

# Energia total por setor (tep x 1000)
E_total = E.sum(axis=0).rename("Energia_Total_tep1000")

### Emprego Detalhado

In [13]:
emp_df = raw_emprego.copy()
emp_df.columns = raw_emprego.iloc[0]
emp_df = emp_df.iloc[1:].reset_index(drop=True)
emp_df.columns = ["id", "descricao", "Informal", "Formal", "Total"]
emp_df[["Informal","Formal","Total"]] = emp_df[["Informal","Formal","Total"]].astype(float)
emp_df.index = sectors

## COEFICIENTES

In [ ]:
e_coef  = emp_df["Total"] / x          # ocupações / R$ mi
en_coef = E_total / x                  # tep(x1000) / R$ mi

e_coef.name  = "Coef_Emprego"
en_coef.name = "Coef_Energia"

# Saída csv
n = len(sectors)

## MULTIPLICADORES

### Multiplicador de Produção

In [23]:
# Multiplicador de produção (tipo I): soma coluna de L

M_prod = L.sum(axis=0).rename("Mult_Producao_TipoI")

df_multi_producao = pd.DataFrame({
    "ID":         range(1, n + 1),
    "Setor":      sector_names,
    "M_Producao": M_prod.values,
}).sort_values("M_Producao", ascending=False).reset_index(drop=True)

df_multi_producao.index = range(1, len(df_multi_producao) + 1)

res_df_multi_producao = df_multi_producao.sort_values("M_Producao", ascending=False)

# arredondar para 2 casas e formato brasileiro (ponto → vírgula)
res_df_multi_producao["M_Producao"] = res_df_multi_producao["M_Producao"].map(lambda x: f"{x:.2f}".replace(".", ","))

res_df_multi_producao.head(10).to_csv("..\\data\\cenarios\\base\\cen_base_multi_producao.csv", index=False, sep=";", encoding="utf-8-sig")

print("🔹 Setores com os Maiores Multiplicadores de Produção:")
print(res_df_multi_producao.head(10)[["ID", "Setor", "M_Producao"]].to_string(index=False))

🔹 Setores com os Maiores Multiplicadores de Produção:
 ID                                                                      Setor M_Producao
 21                                                                  Coquerias       3,53
  8   Abate e produtos de carne, inclusive os produtos do laticínio e da pesca       2,48
  9                                              Fabricação e refino de açúcar       2,38
 22                                              Fabricação de biocombustíveis       2,32
 19                                                      Derivados de petróleo       2,30
 10                                                Outros produtos alimentares       2,30
 20                                                                  Biodiesel       2,29
 35                 Fabricação de automóveis, caminhões e ônibus, exceto peças       2,27
 25 Fabricação de produtos de limpeza, cosméticos/perfumaria e higiene pessoal       2,14
 12                                           

### Multiplicador de Emprego

In [24]:
# Multiplicador de emprego: e_coef . L
M_emp = e_coef.values @ L.values
M_emp = pd.Series(M_emp, index=sectors, name="Mult_Emprego")

df_multi_emprego = pd.DataFrame({
    "ID":          range(1, n + 1),
    "Setor":       sector_names,
    "Coef_Emprego": e_coef.values,
    "M_Emprego":   M_emp.values,
}).sort_values("M_Emprego", ascending=False).reset_index(drop=True)

df_multi_emprego.index = range(1, len(df_multi_emprego) + 1)

res_df_multi_emprego = df_multi_emprego.sort_values("M_Emprego", ascending=False)

# arredondar para 2 casas e formato brasileiro (ponto → vírgula)
res_df_multi_emprego["Coef_Emprego"] = res_df_multi_emprego["Coef_Emprego"].map(lambda x: f"{x:.2f}".replace(".", ","))
res_df_multi_emprego["M_Emprego"]    = res_df_multi_emprego["M_Emprego"].map(lambda x: f"{x:.2f}".replace(".", ","))

res_df_multi_emprego.to_csv("..\\data\\cenarios\\base\\cen_base_multi_emprego.csv", index=False, sep=";", encoding="utf-8-sig")

print("🔹 Setores com os Maiores Multiplicadores de Emprego:")
print(res_df_multi_emprego.head(10)[["ID", "Setor", "Coef_Emprego", "M_Emprego"]].to_string(index=False))

🔹 Setores com os Maiores Multiplicadores de Emprego:
 ID                                                                    Setor Coef_Emprego M_Emprego
 73                                                      Serviços domésticos        88,16     88,16
  2                                   Pecuária, inclusive o apoio à pecuária        38,77     45,94
 14                         Confecção de artefatos do vestuário e acessórios        25,24     33,52
 72                     Organizações associativas e outros serviços pessoais        26,19     31,89
 71                        Atividades artísticas, criativas e de espetáculos        27,82     31,68
 53                                                              Alimentação        20,30     26,97
  3                                  Produção florestal; pesca e aquicultura        21,15     24,54
  8 Abate e produtos de carne, inclusive os produtos do laticínio e da pesca         2,62     23,62
 68                                            

### Multiplicador de Renda

In [25]:
# Coeficiente de VA (VA por unidade de produção)
v_coef = VA["VAB_PIB"] / x

# Multiplicador de VA (PIB): v_coef . L
# v_j = VAB_j / x_j  →  quanto de VA é gerado na economia por R$1 de demanda final no setor j
M_va = v_coef.values @ L.values

M_va = pd.Series(M_va, index=sectors, name="Mult_VA")

df_multi_va = pd.DataFrame({
    "ID":      range(1, n + 1),
    "Setor":   sector_names,
    "Coef_VA": v_coef.values,
    "M_VA":    M_va.values,
}).sort_values("M_VA", ascending=False).reset_index(drop=True)

df_multi_va.index = range(1, len(df_multi_va) + 1)

res_df_multi_va = df_multi_va.sort_values("M_VA", ascending=False)

# arredondar para 2 casas e formato brasileiro (ponto → vírgula)
res_df_multi_va["Coef_VA"] = res_df_multi_va["Coef_VA"].map(lambda x: f"{x:.2f}".replace(".", ","))
res_df_multi_va["M_VA"]    = res_df_multi_va["M_VA"].map(lambda x: f"{x:.2f}".replace(".", ","))

res_df_multi_va.head(10).to_csv("..\\data\\cenarios\\base\\cen_base_multi_pib.csv", index=False, sep=";", encoding="utf-8-sig")

print("🔹 Setores com os Maiores Multiplicadores de PIB:")
print(res_df_multi_va.head(10)[["ID", "Setor", "Coef_VA", "M_VA"]].to_string(index=False))

🔹 Setores com os Maiores Multiplicadores de PIB:
 ID                                                             Setor Coef_VA M_VA
 73                                               Serviços domésticos    1,00 1,00
 59                                           Atividades imobiliárias    0,92 0,98
 67                                                  Educação pública    0,84 0,95
 65                Atividades de vigilância, segurança e investigação    0,81 0,95
 66                 Administração pública, defesa e seguridade social    0,72 0,93
 60 Atividades jurídicas, contábeis, consultoria e sedes de empresas     0,69 0,93
  3                           Produção florestal; pesca e aquicultura    0,75 0,92
 58      Intermediação financeira, seguros e previdência complementar    0,65 0,92
 68                                                  Educação privada    0,72 0,92
 64       Outras atividades administrativas e serviços complementares    0,68 0,90


### Encadeamentos (backward & forward linkages)

In [18]:
# Rasmussen-Hirschman
L_mean = L.values.mean()

# Backward: média coluna / média geral
BL = L.mean(axis=0) / L_mean
BL.name = "Backward_Linkage"

# Forward (Ghosh): H = (I - F)^-1  onde F = diag(x)^-1 . Z
F_raw = x_inv[:, np.newaxis] * Z.values
H_raw = np.linalg.inv(I - F_raw)
H = pd.DataFrame(H_raw, index=sectors, columns=sectors)
FL = H.mean(axis=1) / H.values.mean()
FL.name = "Forward_Linkage"